In [0]:
%sql
create database if not exists delta_training_db;
use delta_training_db;

In [0]:
customers_data = [
    (1, "Aarav Sharma", "Hyderabad", "Gold", 25000),
    (2, "Priya Reddy", "Bengaluru", "Silver", 18000),
    (3, "Rohan Mehta", "Mumbai", "Gold", 32000),
    (4, "Sneha Iyer", "Chennai", "Bronze", 12000),
    (5, "Kiran Patel", "Hyderabad", "Silver", 22000),
    (6, "Ananya Das", "Kolkata", "Gold", 41000),
    (7, "Vikram Singh", "Delhi", "Bronze", 9000),
    (8, "Meera Nair", "Bengaluru", "Silver", 26000)
]

columns = ["customer_id", "customer_name", "city", "membership", "total_spend"]

customers_df = spark.createDataFrame(customers_data, columns)

display(customers_df)

customer_id,customer_name,city,membership,total_spend
1,Aarav Sharma,Hyderabad,Gold,25000
2,Priya Reddy,Bengaluru,Silver,18000
3,Rohan Mehta,Mumbai,Gold,32000
4,Sneha Iyer,Chennai,Bronze,12000
5,Kiran Patel,Hyderabad,Silver,22000
6,Ananya Das,Kolkata,Gold,41000
7,Vikram Singh,Delhi,Bronze,9000
8,Meera Nair,Bengaluru,Silver,26000


In [0]:
%sql
create or replace table customer_delta_sql (
  customer_id long,
  customer_name string,
  city string,
  membership string,
  total_spend int
)
using delta;

In [0]:
%sql
--TRUNCATE TABLE customer_delta_sql;

In [0]:
%sql
insert into customer_delta_sql values
(1, 'John Doe', 'New York', 'Gold', 25000),
(2, 'Jane Smith', 'Los Angeles', 'Silver', 18000),
(3, 'Mike Johnson', 'Chicago', 'Bronze', 32000)

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql 
select * from customer_delta_sql;


customer_id,customer_name,city,membership,total_spend
1,John Doe,New York,Gold,25000
2,Jane Smith,Los Angeles,Silver,18000
3,Mike Johnson,Chicago,Bronze,32000


In [0]:
customers_df_write = customers_df.write.mode("overwrite").saveAsTable("customers_delta_df")

In [0]:
%sql
select * from customers_delta_df;

customer_id,customer_name,city,membership,total_spend
1,Aarav Sharma,Hyderabad,Gold,25000
2,Priya Reddy,Bengaluru,Silver,18000
3,Rohan Mehta,Mumbai,Gold,32000
4,Sneha Iyer,Chennai,Bronze,12000
5,Kiran Patel,Hyderabad,Silver,22000
6,Ananya Das,Kolkata,Gold,41000
7,Vikram Singh,Delhi,Bronze,9000
8,Meera Nair,Bengaluru,Silver,26000


In [0]:
delta_path= "/Files/delta/customers_path_table"
customers_df.write.format("delta").mode("overwrite").save(delta_path)


In [0]:
customers_df = spark.read.format("delta").load(delta_path)
display(delta_path)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4760642765228703>, line 1
----> 1 customers_df = spark.read.format("delta").load(delta_path)
      2 display(delta_path)

NameError: name 'delta_path' is not defined

In [0]:
%sql
CREATE OR REPLACE TABLE retail_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO retail_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed');

num_affected_rows,num_inserted_rows
6,6


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW new_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed')
AS new_orders(order_id,customer_name,city,product,quantity,price,order_status);

In [0]:
%sql
MERGE INTO retail_orders AS target
USING new_orders AS source
ON target.order_id = source.order_id
 
WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status
 
WHEN NOT MATCHED THEN
INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)
 
VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


In [0]:
%sql
-- 1. Insert 2 new orders
INSERT INTO retail_orders VALUES
(107,'Vikram Singh','Delhi','Laptop',2,76000,'Placed'),
(108,'Neha Kapoor','Pune','Mobile',1,29000,'Placed');



num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
-- 2. Update price of product Laptop to 78000
UPDATE retail_orders
SET price = 78000
WHERE product = 'Laptop';



num_affected_rows
3


In [0]:
%sql
-- 3. Delete orders where quantity = 1
DELETE FROM retail_orders
WHERE quantity = 1;


num_affected_rows
5


In [0]:
%sql
-- 4. Create temp table incoming_orders
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(111,'Arjun Nair','Kochi','Tablet',3,25000,'Placed'),
(112,'Meera Joshi','Jaipur','Mobile',2,28000,'Placed')
AS incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);


In [0]:
%sql
-- 5. Merge incoming_orders into retail_orders
MERGE INTO retail_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT (
order_id,
customer_name,
city,
product,
quantity,
price,
order_status
)
VALUES (
source.order_id,
source.customer_name,
source.city,
source.product,
source.quantity,
source.price,
source.order_status
);


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2
